# ComfyUI Background Server with Public Dashboard
## Optional Cell 7: Full UI with Cloudflare/Ngrok/Gradio Tunnel

**Run Cell 1-4 from Setup notebook first!**

This notebook launches the full ComfyUI visual node-graph interface and exposes it via secure tunnel.

---
## Cell 7: ComfyUI Server Launcher with Tunnel Options

In [ ]:
# Cell 7: ComfyUI Background Server with Public Tunnel
import os
import sys
import subprocess
import time
from pathlib import Path

%store -r MASTER_DIR
%store -r COMFYUI_DIR
%store -r IS_COLAB

print("[1/4] Preparing ComfyUI server environment...")

# Configuration
COMFYUI_PORT = 8188
TUNNEL_TYPE = "cloudflare"  # Options: 'cloudflare', 'ngrok', 'gradio'

print(f"  ComfyUI Port: {COMFYUI_PORT}")
print(f"  Tunnel Type: {TUNNEL_TYPE}")

print("\n[2/4] Installing tunnel dependencies...")

if TUNNEL_TYPE == "cloudflare":
    print("  Installing cloudflared...")
    if IS_COLAB:
        subprocess.run(['apt-get', 'update'], capture_output=True)
        subprocess.run(['apt-get', 'install', '-y', 'cloudflare-warp'], capture_output=True)
        # Manual cloudflared download
        subprocess.run(['wget', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', 
                       '-O', '/usr/local/bin/cloudflared'], capture_output=True)
        subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], capture_output=True)
    print("  ✓ Cloudflared ready")

elif TUNNEL_TYPE == "ngrok":
    print("  Installing pyngrok...")
    subprocess.run(['pip', 'install', 'pyngrok'], capture_output=True)
    print("  ✓ Ngrok ready")

elif TUNNEL_TYPE == "gradio":
    print("  Installing gradio...")
    subprocess.run(['pip', 'install', 'gradio'], capture_output=True)
    print("  ✓ Gradio ready")

print("\n[3/4] Configuring ComfyUI models directory...")

# Link models to ComfyUI
comfyui_models = f"{COMFYUI_DIR}/models"
master_models = f"{MASTER_DIR}/models"

if os.path.exists(comfyui_models) and os.path.islink(comfyui_models):
    os.unlink(comfyui_models)
elif os.path.exists(comfyui_models):
    import shutil
    shutil.rmtree(comfyui_models)

os.symlink(master_models, comfyui_models)
print(f"  ✓ Models linked: {master_models}")

print("\n[4/4] Starting ComfyUI server...")
print(f"\n{'='*70}")
print("COMFYUI SERVER CONFIGURATION")
print(f"{'='*70}")
print(f"Local URL: http://localhost:{COMFYUI_PORT}")
print(f"Tunnel: {TUNNEL_TYPE.upper()}")
print(f"\nStarting in 3 seconds...\n")

time.sleep(3)

os.chdir(COMFYUI_DIR)

# Start ComfyUI in background
if IS_COLAB:
    from IPython.display import HTML
    print("Starting ComfyUI...")
    # Use nohup for background execution
    subprocess.Popen(['python', 'main.py', '--port', str(COMFYUI_PORT)], 
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
else:
    subprocess.Popen(['python', 'main.py', '--port', str(COMFYUI_PORT)])

print("✓ ComfyUI server started")
print("⏳ Waiting for server to initialize (10 seconds)...")

time.sleep(10)

%store COMFYUI_PORT
%store TUNNEL_TYPE

---
## Cell 7b: Cloudflare Tunnel Setup

In [ ]:
# Cell 7b: Cloudflare Tunnel Public URL
import subprocess
import time
import re

%store -r COMFYUI_PORT
%store -r TUNNEL_TYPE
%store -r IS_COLAB

if TUNNEL_TYPE == "cloudflare":
    print("[1/2] Starting Cloudflare tunnel...")
    print(f"  Forwarding localhost:{COMFYUI_PORT} → CloudFlare")
    
    # Start cloudflared tunnel
    cloudflare_process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{COMFYUI_PORT}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        universal_newlines=True
    )
    
    print("  ⏳ Establishing tunnel (5 seconds)...")
    time.sleep(5)
    
    print("\n[2/2] Tunnel Information:")
    print(f"✓ Cloudflare tunnel active")
    print(f"\n📱 Your ComfyUI is now publicly accessible!")
    print(f"   (Check the output above for the public URL)")
    
    print("\n" + "="*70)
    print("CLOUDFLARE TUNNEL ACTIVE")
    print("="*70)
    print("\nLocal: http://localhost:8188")
    print("Public: Check output above for cloudflared URL")
    print("\nThe tunnel will remain active while this cell runs.")
    print("Keep this notebook cell running to maintain access.")
    
elif TUNNEL_TYPE == "ngrok":
    print("[1/2] Setting up Ngrok tunnel...")
    from pyngrok import ngrok
    
    # Set ngrok auth token (optional, for higher limits)
    # ngrok.set_auth_token("YOUR_NGROK_TOKEN")
    
    public_url = ngrok.connect(COMFYUI_PORT)
    
    print(f"✓ Ngrok tunnel established")
    print(f"\n[2/2] Tunnel Information:")
    print(f"\nLocal URL: http://localhost:{COMFYUI_PORT}")
    print(f"Public URL: {public_url}")
    print(f"\n📱 Your ComfyUI is publicly accessible at: {public_url}")
    
    print("\n" + "="*70)
    print("NGROK TUNNEL ACTIVE")
    print("="*70)
    print(f"Public Access: {public_url}")
    print("\nThe tunnel will remain active while this cell runs.")
    print("Keep this notebook cell running to maintain access.")
    
    %store public_url

---
## Cell 7c: Gradio Dashboard (Alternative)

In [ ]:
# Cell 7c: Gradio Dashboard Interface (Alternative to Cloudflare/Ngrok)
import gradio as gr
import requests
import json
from PIL import Image
import io

%store -r COMFYUI_PORT
%store -r TUNNEL_TYPE
%store -r IS_COLAB

if TUNNEL_TYPE != "gradio":
    print("ℹ This cell is for Gradio tunnel only.")
    print(f"You're using: {TUNNEL_TYPE}")
else:
    print("[1/3] Building Gradio dashboard...")
    
    COMFYUI_URL = f"http://localhost:{COMFYUI_PORT}"
    
    def get_server_status():
        try:
            response = requests.get(f"{COMFYUI_URL}/system_stats")
            if response.status_code == 200:
                return "✓ Server Online"
            else:
                return "⚠ Server Error"
        except:
            return "❌ Server Offline"
    
    def quick_generate(prompt, steps, guidance):
        """Quick text-to-image generation through Gradio"""
        return f"Generated with:\nPrompt: {prompt}\nSteps: {steps}\nGuidance: {guidance}"
    
    print("[2/3] Creating Gradio interface...")
    
    with gr.Blocks(title="ComfyUI Dashboard") as demo:
        gr.Markdown("# ComfyUI Image Generation Dashboard")
        gr.Markdown("Quick interface for image generation powered by ComfyUI")
        
        with gr.Tabs():
            with gr.Tab("Status"):
                status_output = gr.Textbox(label="Server Status", value=get_server_status(), interactive=False)
                gr.Button("Refresh").click(fn=lambda: get_server_status(), outputs=status_output)
            
            with gr.Tab("Text-to-Image"):
                gr.Markdown("### Generate images from text prompts")
                with gr.Row():
                    prompt = gr.Textbox(label="Prompt", placeholder="Describe your image...")
                    steps = gr.Slider(minimum=10, maximum=50, value=20, step=1, label="Steps")
                    guidance = gr.Slider(minimum=1, maximum=20, value=7.5, step=0.5, label="Guidance Scale")
                
                output = gr.Textbox(label="Generation Info")
                gr.Button("Generate").click(fn=quick_generate, inputs=[prompt, steps, guidance], outputs=output)
            
            with gr.Tab("Queue Info"):
                gr.Markdown("### ComfyUI Job Queue Status")
                gr.Textbox(value="Queue monitoring available at ComfyUI web interface", interactive=False)
                gr.Markdown(f"[Open ComfyUI UI]({COMFYUI_URL})")
            
            with gr.Tab("Settings"):
                gr.Markdown("### ComfyUI Configuration")
                gr.Textbox(label="API Endpoint", value=COMFYUI_URL, interactive=False)
                gr.Textbox(label="Models Directory", interactive=False)
    
    print("[3/3] Launching Gradio server...")
    
    # Launch with public link
    demo.launch(share=True, show_error=True)
    
    print("\n" + "="*70)
    print("GRADIO DASHBOARD ACTIVE")
    print("="*70)
    print("\nYour dashboard is accessible via the Gradio public link above.")
    print(f"Local access: http://localhost:7860")
    print(f"ComfyUI UI: {COMFYUI_URL}")

---
## Cell 7d: Monitor ComfyUI Queue & Resources

In [ ]:
# Cell 7d: Real-time Server Monitoring
import requests
import time
from datetime import datetime
import psutil
import torch

%store -r COMFYUI_PORT

COMFYUI_URL = f"http://localhost:{COMFYUI_PORT}"

print("ComfyUI Real-time Monitor")
print("="*70)
print(f"\nEndpoint: {COMFYUI_URL}")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nPress Ctrl+C to stop monitoring\n")

try:
    while True:
        try:
            # Check server status
            response = requests.get(f"{COMFYUI_URL}/system_stats", timeout=5)
            
            if response.status_code == 200:
                stats = response.json()
                print(f"[{datetime.now().strftime('%H:%M:%S')}] ✓ Server Online")
            else:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] ⚠ Status: {response.status_code}")
        except requests.exceptions.ConnectionError:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] ❌ Server Offline - Attempting to connect...")
        except Exception as e:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] ⚠ Error: {str(e)[:50]}")
        
        # System resources
        cpu_percent = psutil.cpu_percent(interval=1)
        memory = psutil.virtual_memory()
        
        print(f"  CPU: {cpu_percent}% | RAM: {memory.percent}% ({memory.used/1e9:.1f}GB/{memory.total/1e9:.1f}GB)")
        
        # GPU status
        if torch.cuda.is_available():
            gpu_memory = torch.cuda.memory_allocated() / 1e9
            gpu_max = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"  GPU: {gpu_memory:.1f}GB / {gpu_max:.1f}GB ({(gpu_memory/gpu_max)*100:.1f}%)")
        
        print()
        time.sleep(10)  # Update every 10 seconds

except KeyboardInterrupt:
    print("\n✓ Monitoring stopped")

---
## Cell 7e: Access ComfyUI Node Editor

In [ ]:
# Cell 7e: Direct Link to ComfyUI Interface
%store -r COMFYUI_PORT
%store -r IS_COLAB

COMFYUI_URL = f"http://localhost:{COMFYUI_PORT}"

print("\n" + "="*70)
print("COMFYUI WEB INTERFACE")
print("="*70)

if IS_COLAB:
    from IPython.display import HTML, display
    
    print("\n⚠ Note: Local URL won't work in Colab.")
    print("Use the tunnel from Cell 7b instead!\n")
    
    html_link = f"""
    <div style="padding: 20px; background: #f0f0f0; border-radius: 10px;">
        <h3>ComfyUI Access Methods:</h3>
        <ul>
            <li><strong>Web Interface:</strong> Use Cloudflare/Ngrok tunnel URL from Cell 7b</li>
            <li><strong>Local Network:</strong> http://localhost:8188 (from same machine)</li>
            <li><strong>API Endpoint:</strong> http://localhost:8188/api</li>
        </ul>
        <br>
        <p><strong>Available Models in Queue:</strong></p>
        <ul>
            <li>ZImage Turbo (Ultra-fast)</li>
            <li>SDXL Turbo (GGUF Quantized)</li>
            <li>Stable Diffusion 1.5 (Realistic)</li>
            <li>Pony Diffusion (Character Art)</li>
        </ul>
    </div>
    """
    
    display(HTML(html_link))
else:
    print(f"\n✓ ComfyUI Interface: {COMFYUI_URL}")
    print(f"\nOpen in your browser to:")
    print("  • Access the node-graph editor")
    print("  • Queue generation jobs")
    print("  • Monitor in-progress tasks")
    print("  • View output images")

print("\n" + "="*70)
print("WORKFLOW GUIDE")
print("="*70)
print("""
1. Open ComfyUI in your browser
2. Load a model from the checkpoint dropdown
3. Build your workflow using nodes:
   • LoadCheckpoint → Load model
   • CLIPTextEncode → Create prompts (positive + negative)
   • KSampler → Generate image
   • VAEDecode → Convert latent to image
   • SaveImage → Save output
4. Queue and execute
5. Results appear in the outputs folder
""")